In [9]:
import os 
from groq import Groq
import numpy as np 
from dotenv import load_dotenv 

load_dotenv()
client=Groq(api_key=os.getenv("GROQ_API_KEY"))


In [10]:
models=client.models.list()
for m in models.data :
    print (m.id)



openai/gpt-oss-20b
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
canopylabs/orpheus-arabic-saudi


In [11]:
model = "openai/gpt-oss-120b"

In [12]:
text_prompt="should oxford commas always be used?"

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": text_prompt}
    ]
)

print(response.choices[0].message.content)


**Short answer:**  
No—there’s no universal rule that the Oxford (or “serial”) comma must be used in every sentence, but many style guides and most writers recommend it because it usually makes a list clearer. Whether you include it should depend on the style guide you’re following and, when no guide is imposed, on what makes the sentence easiest to read.

---

## What the Oxford comma is
In a list of three or more items, the Oxford comma is the **final comma placed before the coordinating conjunction** (usually *and* or *or*).

> I bought apples, oranges, **and** bananas.  

Without the Oxford comma:

> I bought apples, oranges **and** bananas.

---

## Major style‑guide positions

| Style guide | Recommendation | Typical domains |
|-------------|----------------|-----------------|
| **Chicago Manual of Style (CMOS)** | **Strongly recommends** the Oxford comma in all series. | Academic publishing, books, most non‑journalistic writing. |
| **APA (American Psychological Association)** |

##Temperature Comparison 

In [13]:
prompt2="Describe a sunset over a neon cyberpunk city in 2 sentences."

low_temp_response= client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt2}
    ],
    temperature=0.0

)

print("Low temperature response:")
print(low_temp_response.choices[0].message.content)

Low temperature response:
The dying sun bleeds magenta and amber across the sky, its light refracting off towering glass spires and casting long, trembling shadows that dance between the flickering holographic billboards. Below, the city erupts in a riot of neon—electric blues, hot pinks, and acid greens—while rain‑slick streets mirror the celestial blaze, turning the metropolis into a living, pulsing canvas of light and chrome.


In [14]:
high_temp_response= client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt2}
    ],
    temperature=1.0
)
print("High temperature response:")
print(high_temp_response.choices[0].message.content)


High temperature response:
A molten orange sun sinks behind a jagged skyline, painting the clouds in bruised violet and molten gold before the artificial lights flicker on like restless fireflies. Below, towering holographic billboards blaze in electric pink and teal, their neon veins weaving a luminous tapestry that swallows the fading twilight and turns the city into a pulsing, technicolor dream.


In [15]:
token_response=client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt2}
    ],\
    max_tokens=50,
    seed=42
)

print(token_response.choices[0].message.content)


A blood‑orange sun dips behind towering megastruct


Summarize 


In [16]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"
response_summarize = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"system", "content":"You are a helpful assistant."},
        {"role":"user", "content": prompt}
    ],
    temperature=0.7,
    seed=42
)

print(response_summarize.choices[0].message.content)

**TL;DR:** Scaling up language models dramatically boosts their ability to perform new tasks from just a few examples—often matching or surpassing the performance of models that were fine‑tuned on large, task‑specific datasets.


In [18]:
import numpy as np

def cosine_similarity(a,b):
    a=np.array(a)
    b=np.array(b)
    return np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b))

In [19]:
from sentence_transformers import SentenceTransformer

embed_model=SentenceTransformer("all-MiniLM-L6-v2")

text="the quick brown fox jumped over  lazy dog"
vector=embed_model.encode(text)

print(f"Embedding vector shape: {vector.shape}")
print(f"First 5 vector values: {vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding vector shape: (384,)
First 5 vector values: [0.03437024 0.08424441 0.04558147 0.06299912 0.0517195 ]


In [20]:
automodbile_embedding = embed_model.encode("automobile")
vehicle_embedding = embed_model.encode("automobile")
dinosaur_embedding = embed_model.encode("automobile")
stick_embedding = embed_model.encode("automobile")

print("automobile vs vehicle",cosine_similarity(automodbile_embedding,vehicle_embedding))
print("automobile vs dinosaur",cosine_similarity(automodbile_embedding,dinosaur_embedding))
print("automobile vs stick",cosine_similarity(automodbile_embedding,dinosaur_embedding))

automobile vs vehicle 1.0000001
automobile vs dinosaur 1.0000001
automobile vs stick 1.0000001


In [21]:
documents = [
    " Carlos Alberto joined Werder Bremen for a record fee of 7.8 million euros.",
    " David Beckham moved to the US Major League Soccer playing for LA Galaxy.",
]

doc_embedding=embed_model.encode(documents)

query="Which team did Carlos Alberto join?"

query_embedding=embed_model.encode(query)

similarities=[cosine_similarity(query_embedding,doc_emb) for doc_emb in doc_embedding] 
best_doc_idx=np.argmax(similarities)
most_relevant_doc=documents[best_doc_idx]

response_next_word=client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": f"Answer using this context: {most_relevant_doc}"},
        {"role": "user", "content": query}
    ]
)

print(response_next_word.choices[0].message.content)

Carlos Alberto joined **Werder Bremen**.
